# 07 — State Preparation and Visual Checks

A critical skill in quantum computing is **verifying** that your circuit actually prepares the state you intended.

This notebook shows how to:
- Prepare arbitrary single-qubit states
- Use the statevector simulator to get the exact state (no shot noise)
- Verify the Bloch vector matches the expected geometry
- Use visual checks as a sanity tool


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from helpers.notebook_style import setup_notebook
from helpers.plotting import draw_bloch_sphere
from helpers.display_utils import show_state_vector, show_info_table
setup_notebook()

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

def get_statevector(qc):
    """Use the statevector simulator to get the exact state vector."""
    sim = AerSimulator(method="statevector")
    qc_sv = qc.copy()
    qc_sv.save_statevector()
    job = sim.run(qc_sv.decompose())
    return np.array(job.result().get_statevector())

def bloch_from_spinor(sv):
    a, b = sv[0], sv[1]
    return np.array([
        2*(a.conjugate()*b).real,
        2*(a.conjugate()*b).imag,
        abs(a)**2 - abs(b)**2,
    ])


## Preparing standard states

Let's verify that our circuits actually produce the intended Bloch vectors.


In [ ]:
def prep_and_verify(label, circuit_fn, expected_bloch):
    """Prepare a state, compute its Bloch vector, compare to expected."""
    qc = QuantumCircuit(1)
    circuit_fn(qc)
    sv = get_statevector(qc)
    bv = bloch_from_spinor(sv)
    err = np.linalg.norm(bv - np.array(expected_bloch))
    status = "✓" if err < 1e-6 else f"✗ (error={err:.2e})"
    print(f"{label:<12}  Bloch=({bv[0]:+.4f}, {bv[1]:+.4f}, {bv[2]:+.4f})  {status}")
    return sv, bv

print(f"{'State':<12}  {'Bloch vector':<42}  {'Check'}")
print("-" * 65)
states_to_check = [
    ("|0⟩",  lambda qc: None,               [0, 0, 1]),
    ("|1⟩",  lambda qc: qc.x(0),            [0, 0, -1]),
    ("|+⟩",  lambda qc: qc.h(0),            [1, 0, 0]),
    ("|-⟩",  lambda qc: (qc.x(0), qc.h(0)),[−1, 0, 0]),
    ("|+i⟩", lambda qc: (qc.h(0), qc.s(0)), [0, 1, 0]),
]
results = {}
for label, circuit_fn, expected in states_to_check:
    sv, bv = prep_and_verify(label, circuit_fn, expected)
    results[label] = bv


In [ ]:
# Visualise all on the Bloch sphere
colours = ["green", "red", "steelblue", "darkorange", "violet"]
vectors = [(bv, label, c) for (label, bv), c in zip(results.items(), colours)]

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection="3d")
draw_bloch_sphere(ax=ax, vectors=vectors, title="Prepared states — visual verification")
plt.tight_layout()
plt.show()


## Arbitrary state preparation with Ry + Rz

Any single-qubit state $|\psi\rangle = \cos(\theta/2)|0\rangle + e^{i\phi}\sin(\theta/2)|1\rangle$  
can be prepared with the decomposition $R_z(\phi)R_y(\theta)|0\rangle$.


In [ ]:
def prepare_arbitrary(theta, phi):
    """Prepare the state with Bloch angles (theta, phi)."""
    qc = QuantumCircuit(1)
    qc.ry(theta, 0)
    qc.rz(phi, 0)
    return qc

# Target: θ = π/3, φ = π/4
theta_target = np.pi / 3
phi_target   = np.pi / 4

expected_bloch = np.array([
    np.sin(theta_target) * np.cos(phi_target),
    np.sin(theta_target) * np.sin(phi_target),
    np.cos(theta_target),
])

qc_arb = prepare_arbitrary(theta_target, phi_target)
sv_arb = get_statevector(qc_arb)
bv_arb = bloch_from_spinor(sv_arb)

print(f"Expected Bloch:  ({expected_bloch[0]:+.4f}, {expected_bloch[1]:+.4f}, {expected_bloch[2]:+.4f})")
print(f"Obtained Bloch:  ({bv_arb[0]:+.4f}, {bv_arb[1]:+.4f}, {bv_arb[2]:+.4f})")
print(f"Error:           {np.linalg.norm(bv_arb - expected_bloch):.2e}")

fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(111, projection="3d")
draw_bloch_sphere(
    ax=ax,
    vectors=[(bv_arb, r"$|\psiangle$", "crimson")],
    title=rf"Arbitrary state: $\theta={theta_target:.2f}$, $\phi={phi_target:.2f}$",
)
plt.tight_layout()
plt.show()


## Summary

- The statevector simulator gives exact amplitudes (no shot noise) — ideal for verification.
- Visual checks on the Bloch sphere quickly reveal state preparation errors.
- Any single-qubit state can be prepared with $R_y(\theta)R_z(\phi)|0\rangle$.
- In `rqm-core`, `Spinor.to_bloch()` and `BlochVector.to_circuit()` handle all of this.

**Next:** [08_gate_composition_as_geometry.ipynb](08_gate_composition_as_geometry.ipynb)
